[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/sensortwin-transformer-agent/blob/master/notebooks/06_label_efficiency_colab.ipynb)

# SensorTwin — label efficiency: does masked pretraining help when labels are scarce?

Runs `scripts/label_efficiency_sweep` (same as `make label-efficiency-full`): masked-patch
pretraining once per seed, then five arms (scratch transformer / pretrained + fine-tune /
pretrained + linear probe / CNN / XGBoost) at 1 / 5 / 10 / 100% label fractions, scored by test
macro-F1.

**Budget parity:** both transformer arms select their learning rate from the same two-point
validation budget (1e-3, 1e-4), so a null result cannot be an artifact of the fine-tune arm
training at a fixed lower LR — the confound in the superseded 2026-06-25 run. An earlier
single-seed run under that confound showed pretraining not helping; this matched-budget,
multi-seed run is the real verdict either way.

Budget: the 100% fraction dominates — expect several hours on a T4 for 3 seeds; the per-seed loop
prints progress. Artifacts land in `reports/experiment_summaries/label_efficiency_summary.{json,md}`.

In [ ]:
# Opened from the Colab badge? Only the notebook is present — clone the public repo, then install.
import os

if not os.path.exists("sensortwin"):
    !git clone https://github.com/jman4162/sensortwin-transformer-agent.git
    %cd sensortwin-transformer-agent
%pip install -q -e ".[ml]"

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
SEEDS = 3            # integer count: seeds 0..N-1
EPOCHS_PRETRAIN = 50
EPOCHS_FINETUNE = 30
OUT = "reports/experiment_summaries"

In [ ]:
from scripts.label_efficiency_sweep import main as sweep

sweep([
    "--mode", "colab_standard",
    "--seeds", str(SEEDS),
    "--epochs-pretrain", str(EPOCHS_PRETRAIN),
    "--epochs-finetune", str(EPOCHS_FINETUNE),
    "--out", OUT,
])

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

display(Markdown(Path(f"{OUT}/label_efficiency_summary.md").read_text()))

In [ ]:
try:
    from google.colab import files

    files.download(f"{OUT}/label_efficiency_summary.md")
    files.download(f"{OUT}/label_efficiency_summary.json")
except Exception as e:
    print("Not in Colab or download unavailable:", e)

## Reading the result

- The summary table is mean ± sample std over seeds; per-seed values are in the JSON.
- The pretraining question is the `pretrained_ft` − `scratch` gap at 1-10% fractions. If it is
  ≈ 0 or negative under this matched budget, the honest verdict is "masked pretraining does not
  improve label efficiency on this benchmark" — commit that, don't tune around it.
- Commit `label_efficiency_summary.{json,md}` so the model card's verdict traces to this run.